# Baseline and Variant Review — Audited v2

## tl;dr

This bounded review tests the model weaknesses in the main Notebook's cells 8–9. All experiments use the audited geographic inputs, the same 16 input fields, the same temporal folds, and the same 7,003 March validation orders. The final test remains locked; stacking is not fitted.

The selected Ridge variant (log transforms of seven nonnegative numeric inputs; alpha 1,000) achieves March MAE **6.85 days**, compared with **6.91** for the original Ridge on v2. Recency-weighted forest achieves **6.91**, compared with **7.01** for the original forest. These are small improvements: systematic underprediction and poor slow-delivery accuracy remain. Full-depth trees fit training data better but generalise worse in temporal CV. A matched boosting loss check does not support automatically replacing the forest.

This is an isolated development experiment, not the final stacking handoff. The existing main Notebook and handoff retain their labelled historical v1 outputs.

## Context & Methods

Simple baselines remain ordinary least squares and a single decision tree. Ridge regularises the linear baseline; forest averages trees. The reference promise/mean/median rules are distinct from these fitted baselines.

Four bounded hypotheses are tested: (1) stronger regularisation and fixed log input transforms can reduce instability; (2) less restrictive forest leaves may fit useful detail; (3) recent orders may be more relevant, tested using an a priori 180-day half-life; (4) MAE-oriented loss may differ from squared loss, tested with a matched histogram-boosting pair within the tree family. Boosting is a diagnostic, not stacking. All 18 configurations are fixed in the source before fitting; configurations are selected by the mean of three training CV MAEs, then inspected on March once per route, with original fixed comparators.

### Key Assumptions

- Purchase-time quote, basket and catalogue/location availability remain archive assumptions. Receipt, final status and reviews are excluded.
- Recency weights use purchase dates before each training cutoff, never March errors. Imputation/encoding/scaling are learned inside each fit; weights affect the estimator, not preprocessing.
- The last training CV fold has zero >60-day orders because scoring is conditional on receipts known by March 1. CV is suitable for this bounded comparison but is not an unbiased estimate of future performance.
- March has already been used in earlier development. Its observed small gains need confirmation on the locked final test after the design is frozen. Selected OOF columns are non-nested and are not a final performance estimate.
- Valid long deliveries are retained. No offset is fitted to March residuals, and no future outcome is added as an input.

Sources: [Ridge objective](https://scikit-learn.org/1.7/modules/generated/sklearn.linear_model.Ridge.html), [forest complexity controls](https://scikit-learn.org/1.7/modules/generated/sklearn.ensemble.RandomForestRegressor.html), [histogram boosting losses](https://scikit-learn.org/1.7/modules/generated/sklearn.ensemble.HistGradientBoostingRegressor.html). Implementation: `src/baseline_variant_review.py`; raw source and geographic audit: `notebooks/data_preparation_audit.ipynb`.

In [1]:
from pathlib import Path
import sys, json
from IPython.display import display
import pandas as pd
ROOT = Path.cwd() if (Path.cwd()/"src").is_dir() else Path.cwd().parent
sys.path.insert(0, str(ROOT/"src"))
import baseline_variant_review as review
specs = review.configurations()
display(pd.DataFrame.from_dict(specs, orient="index"))

,kind,params,log_target,log_inputs,half_life_days,stage
linear,linear,{},False,False,NaN,original_grid
ridge_a10,ridge,{'alpha': 10},False,False,NaN,original_grid
ridge_a100,ridge,{'alpha': 100},False,False,NaN,original_grid
tree_d8,tree,"{'max_depth': 8, 'min_samples_leaf': 20}",False,False,NaN,original_grid
tree_d14,tree,"{'max_depth': 14, 'min_samples_leaf': 20}",False,False,NaN,original_grid
rf_d16_l5,forest,"{'max_depth': 16, 'min_samples_leaf': 5, 'max_...",False,False,NaN,original_grid
rf_d24_l10,forest,"{'max_depth': 24, 'min_samples_leaf': 10, 'max...",False,False,NaN,original_grid
rf_log_d24_l10,forest,"{'max_depth': 24, 'min_samples_leaf': 10, 'max...",True,False,NaN,original_grid
ridge_a1,ridge,{'alpha': 1},False,False,NaN,bounded_review
ridge_a1000,ridge,{'alpha': 1000},False,False,NaN,bounded_review


## Data and Training CV

Input: `versions/preparation_v2/data/eligible_orders.csv`, reconstructed from the fixed course ZIP. Only the 53,644 development training and 7,003 March validation rows enter the experiment. Source hash, seed, features, folds and the predeclared configuration grid are saved before fitting. No data is downloaded.

Setting `RUN_EXPERIMENT = True` reruns training; `False` displays the verified cached outputs. The default below runs the complete experiment.

In [2]:
RUN_EXPERIMENT = True
if RUN_EXPERIMENT:
    eligible, specs, cv, summary, chosen = review.run_cv()
else:
    cv = pd.read_csv(review.DEST/"outputs/tables/cv_results.csv")
    summary = pd.read_csv(review.DEST/"outputs/tables/cv_summary.csv")
    chosen = json.loads((review.DEST/"outputs/cv_selection.json").read_text())["chosen_by_training_CV"]
print("CV choices frozen before March inspection:", chosen)
display(summary.sort_values(["algorithm", "CV_MAE_mean"]).round(3))

CV 1: linear: fit MAE=4.372, score MAE=17.664


CV 1: ridge_a10: fit MAE=4.380, score MAE=4.341


CV 1: ridge_a100: fit MAE=4.404, score MAE=3.933


CV 1: tree_d8: fit MAE=4.342, score MAE=4.131


CV 1: tree_d14: fit MAE=4.106, score MAE=4.289


...

CV 3: rf_recent180: fit MAE=3.672, score MAE=4.235


CV 3: boost_squared: fit MAE=4.240, score MAE=4.176


CV 3: boost_absolute: fit MAE=4.067, score MAE=4.267


CV choices frozen before March inspection: {'linear': 'linear', 'ridge': 'ridge_logx_a1000', 'tree': 'tree_d8', 'forest': 'rf_recent180', 'boost': 'boost_squared'}


,algorithm,stage,model,CV_MAE_mean,CV_MAE_std,CV_RMSE_mean,fit_MAE_mean
1,boost,bounded_review,boost_squared,4.576,0.926,6.951,4.162
0,boost,bounded_review,boost_absolute,4.641,1.045,7.260,3.989
4,forest,bounded_review,rf_recent180,4.545,0.839,6.976,3.653
6,forest,original_grid,rf_d24_l10,4.552,0.845,6.975,3.650
5,forest,original_grid,rf_d16_l5,4.558,0.849,6.981,3.579
3,forest,bounded_review,rf_full_l3,4.602,0.840,7.042,2.677
7,forest,original_grid,rf_log_d24_l10,4.604,0.945,7.237,3.531
2,forest,bounded_review,rf_full_l1,5.071,0.622,7.524,1.992
8,linear,original_grid,linear,9.580,7.091,14.339,4.348
12,ridge,bounded_review,ridge_logx_a1000,4.540,0.810,6.906,4.410


## March Validation Results

Only CV-selected configurations plus the original fixed comparators are inspected. MAE/RMSE are in days; bias is predicted minus actual. A negative bias means underprediction. All rows refer to exactly the same March orders; no model is selected by averaging its training and validation errors.

In [3]:
if RUN_EXPERIMENT:
    comparison, predictions = review.evaluate_selected(eligible, specs, chosen)
else:
    comparison = pd.read_csv(review.DEST/"outputs/tables/model_comparison.csv")
    predictions = pd.read_csv(review.DEST/"outputs/predictions/validation_predictions.csv")
display(comparison.round(3))
print("Forest paired comparison:")
display(json.loads((review.DEST/"outputs/paired_forest_comparison.json").read_text()))

March linear: MAE=6.918, bias=-4.329


March ridge_a100: MAE=6.913, bias=-4.314


March tree_d8: MAE=7.236, bias=-4.534


March rf_d24_l10: MAE=7.011, bias=-4.350


March ridge_logx_a1000: MAE=6.847, bias=-4.193


March rf_recent180: MAE=6.913, bias=-4.207


March boost_squared: MAE=6.916, bias=-4.396


,model,role,n,MAE_days,RMSE_days,R2,bias_days,P90_abs_error_days,tail_n,tail_MAE_days,fit_MAE_days,within_3_days_pct
0,promise,reference,7003,9.825,12.202,-0.135,5.732,18.840,37,40.754,NaN,NaN
1,train_mean,reference,7003,8.407,11.943,-0.087,-3.384,19.268,37,56.479,NaN,NaN
2,train_median,reference,7003,8.645,12.594,-0.209,-5.238,21.122,37,58.334,NaN,NaN
3,linear,CV_selected,7003,6.918,10.674,0.131,-4.329,17.091,37,52.928,4.777,39.769
4,ridge_a100,fixed_original_comparator,7003,6.913,10.663,0.133,-4.314,17.042,37,52.898,4.778,39.626
5,tree_d8,CV_selected,7003,7.236,11.205,0.043,-4.534,18.457,37,53.236,4.738,40.283
6,rf_d24_l10,fixed_original_comparator,7003,7.011,10.840,0.104,-4.350,17.691,37,52.314,3.963,40.383
7,ridge_logx_a1000,CV_selected,7003,6.847,10.584,0.146,-4.193,16.906,37,53.019,4.783,39.640
8,rf_recent180,CV_selected,7003,6.913,10.682,0.130,-4.207,17.347,37,52.132,3.990,40.326
9,boost_squared,CV_selected,7003,6.916,10.753,0.119,-4.396,17.406,37,52.556,4.586,40.440


Forest paired comparison:


{'candidate': 'rf_recent180',
 'comparator': 'rf_d24_l10',
 'MAE_difference_days': -0.09769326533154135,
 'day_cluster_bootstrap_95_interval': [-0.11608983106526312,
  -0.07949070119462309],
 'seed': 5006,
 'resamples': 2000,
 'interpretation': 'negative favors candidate; descriptive on an already-used validation month, not an unbiased final-test inference'}

## Error Diagnostics

Training target mean is 12.92 days versus March 16.30 days. This establishes a period difference in observed completed deliveries, not its cause. Current models fail most noticeably on slow deliveries. Outcome-duration groups are diagnostics only; they are never inputs or a rule for deleting orders.

A deep forest's much lower training error accompanied by higher temporal CV error indicates overfitting. Stronger regularisation and recency weighting improve the bounded results slightly, while squared/absolute boosting losses show a tradeoff rather than a universal solution.

In [4]:
display(pd.read_csv(review.DEST/"outputs/tables/period_targets.csv").round(3))
durations = pd.read_csv(review.DEST/"outputs/tables/duration_errors.csv")
display(durations.loc[durations.model.isin([chosen["ridge"], chosen["forest"]]), ["model", "duration_group", "n", "MAE_days", "bias_days"]].round(3))
for name in [chosen["ridge"], chosen["forest"]]:
    error = (predictions[name]-predictions.lead_time_days).abs()
    slow = predictions.lead_time_days > 30
    print(name, "orders >30 days:", int(slow.sum()), "share of total absolute error:", round(error[slow].sum()/error.sum()*100, 2), "%")

,period,n,mean_days,median_days,tail_n
0,training,53644,12.917,11.063,124
1,March validation,7003,16.301,13.368,37
2,CV 1 scoring,12215,11.528,10.108,18
3,CV 2 scoring,17260,14.274,12.084,44
4,CV 3 scoring,9970,12.451,11.169,0


,model,duration_group,n,MAE_days,bias_days
20,ridge_logx_a1000,"(-0.001, 7.0]",1404,3.024,2.610
21,ridge_logx_a1000,"(7.0, 14.0]",2248,3.041,1.258
22,ridge_logx_a1000,"(14.0, 30.0]",2507,6.626,-6.221
23,ridge_logx_a1000,"(30.0, 60.0]",807,22.674,-22.674
24,ridge_logx_a1000,"(60.0, inf]",37,53.019,-53.019
25,rf_recent180,"(-0.001, 7.0]",1404,2.704,2.492
26,rf_recent180,"(7.0, 14.0]",2248,3.110,0.847
27,rf_recent180,"(14.0, 30.0]",2507,7.196,-6.101
28,rf_recent180,"(30.0, 60.0]",807,21.878,-21.857
29,rf_recent180,"(60.0, inf]",37,52.132,-52.132


ridge_logx_a1000 orders >30 days: 844 share of total absolute error: 42.25 %
rf_recent180 orders >30 days: 844 share of total absolute error: 40.45 %


## Takeaways

The original implementation is runnable and avoids outcome leakage, but its variants are not yet a strong forecasting solution. Keep the simple models as comparisons; the stronger Ridge and recency forest are candidate revisions, not proof that the task is solved. Do not report a practically meaningful Ridge gain from the old six-minute MAE difference.

Further improvement should first investigate purchase-time temporal and route information (for example season/calendar representation, quoted-promise–distance interactions, and genuinely available lagged delivery history). These are hypotheses requiring separate fold-safe feature experiments, not results established here. Reassess label-maturity sensitivity using a separately defined historical backtest before relying on the approximately 4.5-day CV MAE. Do not silently change the current March/test boundaries. Fix the model design before regenerating the canonical handoff and undertaking stacking/final testing.

AI assisted implementation and review. Findings are grounded in the executed experiment; the project remains responsible for final review and submission.

## Reproducibility Checks

The independent check recomputes MAE/RMSE, reloads saved models, checks fit-only imputation, verifies CV/OOF IDs and temporal boundaries, and verifies that the original private handoff files are byte-identical. It does not score final test outcomes.

In [5]:
sys.path.insert(0, str(ROOT/"scripts"))
from verify_baseline_review import verify
validation_audit = verify()

{
  "passed": true,
  "configurations": 18,
  "folds": 3,
  "validation_n": 7003,
  "selected_oof_rows_per_model": 39445,
  "metrics_recomputed": true,
  "pipelines_reloaded": true,
  "legacy_handoff_hashes_unchanged": true,
  "test_scored": false,
  "caveat": "The chronological CV scoring cohort is conditioned on labels known by March 1; March is development evidence, not a final test."
}
